# Qwen3-8B server on Colab (Ollama + Cloudflare quick tunnel)

Project: *Graph-Enhanced Language Models for Scientific Literature*.

1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Copy the printed URL into your local `.env` as `LLM_BASE_URL=<url>/v1`, `LLM_MODEL=qwen3:8b`.

**The tunnel URL is public while this runs. Do not share it. Stop the runtime when done.** Check Colab's terms of use for running servers before relying on this.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install Ollama (official install script)
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start the Ollama server in the background
import subprocess, os, time
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_ORIGINS='*')
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
print(open('ollama.log').read()[-1000:])

In [ ]:
# 3. Download the model (~5 GB, a few minutes). Change to qwen3:4b if memory is short.
MODEL = 'qwen3:8b'
!ollama pull {MODEL}
!ollama list

In [ ]:
# 4. Local test of the OpenAI-compatible endpoint
import requests, json
r = requests.post('http://localhost:11434/v1/chat/completions', json={
    'model': MODEL, 'temperature': 0,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: READY /no_think'}]}, timeout=300)
print(r.status_code, r.json()['choices'][0]['message']['content'])

In [ ]:
# 5. Install cloudflared and open a quick tunnel (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--http-host-header', 'localhost:11434'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(200):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\nPut these lines in your local .env:\n')
print(f'LLM_BACKEND=openai_compatible\nLLM_BASE_URL={url}/v1\nLLM_MODEL={MODEL}')

In [ ]:
# 6. Keep this cell running to keep the session alive. Interrupt it (and stop the runtime) when done.
import time
while True:
    time.sleep(60)
    print('alive', time.strftime('%H:%M:%S'), '| server running:', server.poll() is None)